# 第8章 Element-Wise：逐元素算子操作手册

**仓库定位器**：本 notebook 位于 `hello-gpu/notebooks/part2-kernels/chapter8.ipynb`，对应源码目录 `hello-gpu/code/part2-kernels/chapter8/`，文档位于 `hello-gpu/docs/part2-kernels/chapter8/index.md`。

---

## Goal

本章以 Vector Add 为例，分别用 HIP 和 Triton 实现逐元素算子，验证正确性并测量性能。你将学会：

**核心原则**：correctness before performance（正确性优先于性能）

- 理解 Element-Wise 算子的数据依赖特征
- 掌握 HIP 线程级范式与 Triton 分块范式的编程差异
- 运行边界正确性检查（N=31, 32, 33, 1027 等）
- 执行 GPU event 计时并计算逻辑有效带宽
- 分析 kernel trace 资源字段
- 理解受控访存实验的设计与结果

---

## Prerequisite

- 已完成第 4-7 章：GPU 环境、可信计时、kernel trace、Roofline 基础
- 熟悉 Python 与基础 C++
- 已安装 ROCm 7.13+ 与 PyTorch ROCm 版本
- 理解 FP32 浮点运算与内存带宽概念

---

## Platform

**基线平台**：AMD Radeon RX 9070 XT (gfx1201) + ROCm 7.13 + 原生 Ubuntu 24.04.4 LTS

**其他平台注意事项**：
- **gfx1100 / gfx1151**：可运行，但性能数据与 gfx1201 基线不可直接比较
- **gfx1201**：本章所有性能数据的参考平台
- 跨平台运行时需修改 `GPU_ARCH` 环境变量并重新编译 HIP 代码

---

## Parameter

### 主要参数

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `SIZE` | 16777216 | 输入向量长度（FP32 元素数） |
| `HIP_BLOCK` | 256 | HIP kernel 的 block size |
| `TRITON_BLOCK` | 1024 | Triton t1 的 BLOCK_SIZE（t0 固定为 256） |
| `WARMUP` | 10 | 预热迭代次数 |
| `REPEAT` | 50 | 正式计时迭代次数 |
| `SEED` | 20260716 | 输入生成随机种子 |
| `INDEPENDENT_RUNS` | 3 | 独立进程复跑次数 |
| `GPU_ARCH` | gfx1201 | 目标 GPU 架构 |

### 边界测试尺寸

- `N=1`：单元素
- `N=31`：小于 wave size (32)
- `N=32`：刚好一个 wave
- `N=33`：超过一个 wave
- `N=255, 256, 257`：block 边界
- `N=1027`：不能被 4 整除，测试 float4 尾部处理

---

## Execution

### 1. 环境准备与路径定位

In [ ]:
import subprocess
import json
import csv
import os
import re
from pathlib import Path

# Robust repo root detection: search cwd and parents
def find_repo_root():
    current = Path.cwd().resolve()
    # Check current and up to 5 parent levels
    for _ in range(6):
        # Look for distinctive repo markers
        if (current / 'code').is_dir() and (current / 'notebooks').is_dir():
            return current
        if (current / '.git').exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    # Fallback: assume cwd is repo root
    return Path.cwd()

repo_root = find_repo_root()
code_dir = repo_root / "code" / "part2-kernels" / "chapter8"
evidence_dir = code_dir / "evidence"

print(f"仓库根目录: {repo_root}")
print(f"代码目录: {code_dir}")
print(f"证据目录: {evidence_dir}")
print(f"\n检查关键文件:")
print(f"  HIP 源码: {(code_dir / 'vector_add_hip.hip').exists()}")
print(f"  Triton 源码: {(code_dir / 'vector_add_triton.py').exists()}")
print(f"  运行脚本: {(code_dir / 'run_all.sh').exists()}")

# Detect GPU architecture
print(f"\n检测 GPU 架构:")
arch = None
try:
    result = subprocess.run(['rocminfo'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        detected = re.findall(r'(?<![0-9A-Za-z])gfx(?:1100|1151|1201)(?![0-9A-Za-z])', result.stdout)
        for candidate in detected:
            if candidate in ['gfx1100', 'gfx1151', 'gfx1201']:
                arch = candidate
                print(f"  检测到架构: {arch}")
                if arch in ['gfx1100', 'gfx1151', 'gfx1201']:
                    print(f"  ✓ 支持的架构")
                    if arch != 'gfx1201':
                        print(f"  ⚠ 注意：本章性能数据基于 gfx1201，{arch} 结果仅供参考")
                else:
                    print(f"  ⚠ 未测试的架构，可能需要调整编译参数")
                break
        if arch is None:
            raise RuntimeError("未检测到支持的架构 gfx1100/gfx1151/gfx1201")
    else:
        print("  ⚠ rocminfo 未找到或执行失败")
except (subprocess.TimeoutExpired, FileNotFoundError):
    print("  ⚠ 无法检测 GPU 架构（rocminfo 不可用）")


### 2. HIP 实现概览

HIP 路线包含 5 个版本，每个版本只改变一个维度：

- **hip-v0**：最简单的一个 thread 处理一个元素
- **hip-v1-contiguous**：受控实验，每个 lane 处理 32 个元素，连续访问
- **hip-v1-strided**：受控实验，每个 lane 处理 32 个元素，跨步访问
- **hip-v2**：Grid-Stride Loop，限制 grid 大小让线程重复工作
- **hip-v3**：使用 `float4` 向量类型 + 标量尾部处理

**关键代码片段**（来自 `vector_add_hip.hip`）：

```cpp
// v0: 最简单版本
__global__ void vector_add_v0(const float* input_a,
                              const float* input_b,
                              float* output,
                              std::size_t size) {
    const std::size_t index =
        static_cast<std::size_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (index < size) {
        output[index] = input_a[index] + input_b[index];
    }
}

// v1-contiguous: 连续访问
// index = base + round * 32 + lane

// v1-strided: 跨步访问
// index = base + lane * 32 + round

// v3: float4 向量化
const float4 a = input_a4[vector_index];
const float4 b = input_b4[vector_index];
output4[vector_index] = make_float4(
    a.x + b.x, a.y + b.y, a.z + b.z, a.w + b.w);
```

**不复制完整代码**：完整实现见 `code/part2-kernels/chapter8/vector_add_hip.hip`（521 行）。

### 3. Triton 实现概览

Triton 路线包含 2 个版本，只改变 `BLOCK_SIZE`：

- **triton-t0**：`BLOCK_SIZE=256`，baseline
- **triton-t1**：`BLOCK_SIZE=1024`，更大的 tile

**核心 kernel**（来自 `vector_add_triton.py`）：

```python
@triton.jit
def vector_add_kernel(
    input_a_ptr,
    input_b_ptr,
    output_ptr,
    size,
    BLOCK_SIZE: tl.constexpr,
):
    program_id = tl.program_id(axis=0)
    offsets = program_id * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    valid = offsets < size

    input_a = tl.load(input_a_ptr + offsets, mask=valid, other=0.0)
    input_b = tl.load(input_b_ptr + offsets, mask=valid, other=0.0)
    tl.store(output_ptr + offsets, input_a + input_b, mask=valid)
```

**关键概念**：
- `program_id`：当前 program 在网格中的编号
- `offsets`：一块逻辑下标，不是单个标量
- `mask`：保护越界位置，`valid=False` 的位置不会产生访问

**不复制完整代码**：完整实现见 `code/part2-kernels/chapter8/vector_add_triton.py`（271 行）。

### 4. 内联可视化 (inline visualization)：Triton program、offsets 与 mask

以 `N=13, BLOCK_SIZE=8` 为例，展示 Triton 如何划分 program 和处理尾部：

| Program ID | Offsets | Mask | 有效元素 | 无效元素 |
|------------|---------|------|----------|----------|
| 0 | [0,1,2,3,4,5,6,7] | [T,T,T,T,T,T,T,T] | 0-7 | 无 |
| 1 | [8,9,10,11,12,13,14,15] | [T,T,T,T,T,F,F,F] | 8-12 | 13-15 |

**解释**：
- Program 0 覆盖 0-7，全部有效
- Program 1 覆盖 8-15，但只有 8-12 有效（N=13）
- `mask=False` 的位置（13-15）不会产生 load/store

**实时可视化**（需要 GPU 环境）：
```bash
cd code/part2-kernels
python chapter8/visualize_triton.py --size 13 --block 8 --launch
# 浏览器打开 http://127.0.0.1:5001
```

**注意**：Triton-viz 使用 CPU interpreter，不测量 GPU 性能，仅用于理解地址与 mask。

### 5. 运行完整实验

**前置条件**：
- 已激活 ROCm 环境（`source activate-rocm.sh`）
- 已设置 `SOURCE_COMMIT` 环境变量

**执行命令**：

In [ ]:
if arch not in {'gfx1100', 'gfx1151', 'gfx1201'}:
    raise RuntimeError("请先运行环境检测单元并确认 GPU 架构")
source_commit = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=repo_root,
    capture_output=True, text=True, check=True,
).stdout.strip()
run_env = os.environ.copy()
run_env.update({
    'SOURCE_COMMIT': source_commit,
    'GPU_ARCH': arch,
    'RUN_TRITON_VIZ': '0',
})
print(f'开始运行 Chapter 8 完整实验：arch={arch}, commit={source_commit[:12]}')
subprocess.run(
    ['bash', str(code_dir / 'run_all.sh')],
    cwd=code_dir.parent, env=run_env, check=True,
)
if (code_dir / 'profile_all.sh').is_file():
    subprocess.run(
        ['bash', str(code_dir / 'profile_all.sh')],
        cwd=code_dir.parent, env=run_env, check=True,
    )
print('实验完成；下面的单元格将读取本次生成的 evidence。')


### 6. 加载并展示正确性结果

In [ ]:
# 加载 summary.csv
summary_csv = evidence_dir / "summary.csv"
if not summary_csv.exists():
    print(f"错误：未找到 {summary_csv}")
    print("请先运行 run_all.sh 生成证据文件")
else:
    print("=" * 80)
    print("正确性矩阵")
    print("=" * 80)
    
    # Read CSV using stdlib
    with open(summary_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    
    # Display correctness columns
    print(f"{'Implementation':<25} {'Runtime':<10} {'Shape':<12} {'Block':<8} {'Grid':<8} {'Correct':<10} {'Max Abs Error':<15}")
    print("-" * 100)
    for row in rows:
        impl = row['implementation']
        runtime = row['runtime']
        shape = row['shape']
        block = row['block']
        grid = row['grid']
        correct = row['correct']
        max_err = row['max_abs_error']
        print(f"{impl:<25} {runtime:<10} {shape:<12} {block:<8} {grid:<8} {correct:<10} {max_err:<15}")
    
    # Check if all passed
    all_correct = all(row['correct'] == 'OK' for row in rows)
    max_error = max(float(row['max_abs_error']) for row in rows)
    print(f"\n全部实现正确性: {'✓ 通过' if all_correct else '✗ 失败'}")
    print(f"最大绝对误差: {max_error:.9g}")


### 7. 加载并展示性能结果

In [ ]:
summary_csv = evidence_dir / "summary.csv"
if not summary_csv.exists():
    print(f"错误：未找到 {summary_csv}")
    print("请先运行 run_all.sh 生成证据文件")
else:
    print("=" * 80)
    print("性能结果（Benchmark）")
    print("=" * 80)
    
    # Read CSV using stdlib
    with open(summary_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    
    # Display performance columns
    print(f"{'Implementation':<25} {'Median (ms)':<15} {'Bandwidth (GB/s)':<18} {'Run Min (ms)':<15} {'Run Max (ms)':<15}")
    print("-" * 90)
    for row in rows:
        impl = row['implementation']
        median = float(row['median_ms'])
        bw = float(row['effective_bandwidth_gbs'])
        run_min = float(row['median_ms_run_min'])
        run_max = float(row['median_ms_run_max'])
        print(f"{impl:<25} {median:<15.6f} {bw:<18.6f} {run_min:<15.6f} {run_max:<15.6f}")
    
    # Key observations
    print("\n关键观察：")
    strided = next(r for r in rows if r['implementation'] == 'hip-v1-strided')
    contiguous = next(r for r in rows if r['implementation'] == 'hip-v1-contiguous')
    ratio = float(strided['median_ms']) / float(contiguous['median_ms'])
    print(f"1. hip-v1-strided 用时约为 hip-v1-contiguous 的 {ratio:.2f}× 倍")
    
    triton_t0 = next(r for r in rows if r['implementation'] == 'triton-t0')
    hip_v0 = next(r for r in rows if r['implementation'] == 'hip-v0')
    diff = abs(float(triton_t0['median_ms']) - float(hip_v0['median_ms']))
    print(f"2. triton-t0 与 hip-v0 性能接近，中位数差异约 {diff:.6f} ms")
    print(f"3. hip-v2 (Grid-Stride) 未超过 hip-v0")
    print(f"4. hip-v3 (float4) 未超过 hip-v2")
    print(f"5. triton-t1 (BLOCK_SIZE=1024) 未超过 triton-t0 (BLOCK_SIZE=256)")


### 8. 加载并展示 Profiling 结果

In [ ]:
profile_csv = evidence_dir / "profile_summary.csv"
if not profile_csv.exists():
    print(f"错误：未找到 {profile_csv}")
    print("请先运行 profile_all.sh 生成 profiling 数据")
else:
    print("=" * 80)
    print("Kernel Trace 资源字段")
    print("=" * 80)
    
    # Read CSV using stdlib
    with open(profile_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    
    # Display profiling columns
    print(f"{'Implementation':<25} {'Dispatches':<12} {'Grid X':<10} {'Workgroup X':<12} {'LDS (B)':<10} {'Scratch (B)':<12} {'VGPR':<8} {'SGPR':<8}")
    print("-" * 110)
    for row in rows:
        impl = row['implementation']
        dispatches = row['trace_dispatches']
        grid = row['trace_grid_size_x']
        wg = row['trace_workgroup_size_x']
        lds = row['trace_lds_bytes']
        scratch = row['trace_scratch_bytes']
        vgpr = row['trace_vgpr_count']
        sgpr = row['trace_sgpr_count']
        print(f"{impl:<25} {dispatches:<12} {grid:<10} {wg:<12} {lds:<10} {scratch:<12} {vgpr:<8} {sgpr:<8}")
    
    print("\n关键观察：")
    print("1. hip-v1-contiguous 与 hip-v1-strided 的 trace 资源字段完全相同")
    print("   - Grid、Workgroup、VGPR、SGPR、LDS、Scratch 均相同")
    print("   - 性能差异（约 6.95×）来自地址访问模式，不是资源配置")
    print("2. triton-t1 的 VGPR 从 8 增加到 24（BLOCK_SIZE 从 256 增加到 1024）")
    print("3. 所有实现的 LDS 和 Scratch 均为 0（本章算子不需要共享内存或溢出）")


### 9. 有效带宽计算与解释

In [ ]:
if summary_csv.exists():
    print("=" * 80)
    print("有效带宽计算")
    print("=" * 80)
    
    # Read CSV using stdlib
    with open(summary_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    
    # Extract parameters from first row
    N = int(rows[0]['shape'])
    dtype_size = 4  # FP32
    
    print(f"输入规模: N = {N:,} 个 FP32 元素")
    print(f"逻辑字节数: 3 × N × 4 Byte = {3 * N * dtype_size:,} Byte")
    print(f"  - 读取 A: {N * dtype_size:,} Byte")
    print(f"  - 读取 B: {N * dtype_size:,} Byte")
    print(f"  - 写回 C: {N * dtype_size:,} Byte")
    print()
    
    print("有效带宽公式:")
    print("  BW_effective = (3 × N × 4 Byte) / (median_ms × 1e-3) / 1e9")
    print()
    
    print("示例计算（hip-v0）:")
    hip_v0 = next(r for r in rows if r['implementation'] == 'hip-v0')
    median = float(hip_v0['median_ms'])
    bw = float(hip_v0['effective_bandwidth_gbs'])
    print(f"  median_ms = {median:.6f} ms")
    print(f"  BW_effective = {3 * N * dtype_size} / ({median:.6f} × 1e-3) / 1e9")
    print(f"               = {bw:.6f} GB/s")
    print()
    
    print("重要说明：")
    print("1. 这是逻辑有效带宽，不等于物理 GDDR6 流量")
    print("2. 适合在相同语义、相同 shape、相同计时范围下比较版本")
    print("3. 物理流量需要硬件计数器或更进一步的分析支持")
    print("4. 当前测量范围是 kernel-only GPU event，不含分配和拷贝")


### 10. 对比参考证据

**参考证据来源**：`code/part2-kernels/chapter8/evidence/` (2026-07-19 curated evidence)

**关键数值对比**：

In [ ]:
# 参考证据（来自项目 evidence）
reference_data = {
    "hip-v0": {"median_ms": 0.336324, "bw_gbs": 598.609044},
    "hip-v1-contiguous": {"median_ms": 0.369584, "bw_gbs": 544.738374},
    "hip-v1-strided": {"median_ms": 2.567170, "bw_gbs": 78.423556},
    "hip-v2": {"median_ms": 0.343484, "bw_gbs": 586.130918},
    "hip-v3": {"median_ms": 0.345224, "bw_gbs": 583.176683},
    "triton-t0": {"median_ms": 0.336204, "bw_gbs": 598.823604},
    "triton-t1": {"median_ms": 0.338504, "bw_gbs": 594.753950},
}

if summary_csv.exists():
    with summary_csv.open(newline="", encoding="utf-8") as handle:
        current_rows = {
            row["implementation"]: row
            for row in csv.DictReader(handle)
        }
    print("=" * 80)
    print("当前运行 vs 参考证据")
    print("=" * 80)
    for implementation, reference in reference_data.items():
        current = current_rows.get(implementation)
        if current is None:
            continue
        current_median = float(current["median_ms"])
        current_bw = float(current["effective_bandwidth_gbs"])
        median_diff_pct = abs(current_median - reference["median_ms"]) / reference["median_ms"] * 100
        bw_diff_pct = abs(current_bw - reference["bw_gbs"]) / reference["bw_gbs"] * 100
        print(f"\n{implementation}:")
        print(f"  当前: {current_median:.6f} ms, {current_bw:.6f} GB/s")
        print(f"  参考: {reference['median_ms']:.6f} ms, {reference['bw_gbs']:.6f} GB/s")
        print(f"  差异: {median_diff_pct:.2f}% (时间), {bw_diff_pct:.2f}% (带宽)")
    print("\n注意：参考证据来自 gfx1201 基线平台；跨平台重点比较相对趋势。")
else:
    print("跳过对比（未找到当前运行数据）")

---

## Expected output / interpretation

### 正确性

- 所有实现的 `correct` 字段应为 `OK`
- `max_abs_error` 应为 `0.0`（FP32 精确加法）
- 边界测试（N=31, 32, 33, 1027）全部通过

### 性能趋势

1. **hip-v1-strided 显著慢于 hip-v1-contiguous**
   - 参考数据：约 6.95× 倍
   - 原因：跨步访问破坏了地址连续性
   - 两者的 trace 资源字段完全相同，差异来自访存模式

2. **hip-v0 与 triton-t0 性能接近**
   - 两者都是最简单的正确实现
   - 中位数差异约 0.00012 ms（在三进程范围内）

3. **负结果**
   - hip-v2 (Grid-Stride) 未超过 hip-v0
   - hip-v3 (float4) 未超过 hip-v2
   - triton-t1 (BLOCK_SIZE=1024) 未超过 triton-t0
   - 这些都是有效结果，说明更少 block 或源码向量化不自动带来收益

### Profiling 解读

- **hip-v1 受控对照**：contiguous 与 strided 的 Grid、Workgroup、VGPR、SGPR、LDS、Scratch 完全相同，性能差异纯粹来自地址顺序
- **triton-t1 资源变化**：VGPR 从 8 增加到 24，可能影响占用率
- **LDS 与 Scratch**：所有实现均为 0，本章算子不需要共享内存或寄存器溢出

### 有效带宽

- 逻辑有效带宽 = (3 × N × 4 Byte) / (median_ms × 1e-3) / 1e9
- hip-v0 约 598.6 GB/s（gfx1201 参考）
- hip-v1-strided 约 78.4 GB/s（受控负例）
- **不等于物理 GDDR6 流量**，需要硬件计数器进一步验证

---

## Pass criteria

完成本章操作后，你应该能够：

1. **正确性**
   - ✓ 所有实现通过边界测试（N=31, 32, 33, 1027）
   - ✓ `max_abs_error = 0.0`
   - ✓ precheck 与 postcheck 均为 OK

2. **性能理解**
   - ✓ 能解释 hip-v1-strided 为何显著慢于 hip-v1-contiguous
   - ✓ 能说明逻辑有效带宽的计算公式与适用范围
   - ✓ 能识别负结果（Grid-Stride、float4、更大 tile 未提速）

3. **Profiling 分析**
   - ✓ 能从 profile_summary.csv 指出受控对照固定了哪些资源字段
   - ✓ 能解释 triton-t1 的 VGPR 变化
   - ✓ 能说明为什么 LDS 和 Scratch 为 0

4. **实验设计**
   - ✓ 能说明为什么 hip-v1 受控对照只改变地址顺序
   - ✓ 能解释为什么需要独立进程复跑（INDEPENDENT_RUNS=3）
   - ✓ 能说明当前结论为什么不能外推到其他硬件或 shape

5. **HIP vs Triton**
   - ✓ 能用表格对比两种范式的关键差异（thread vs program, 标量 vs tile）
   - ✓ 能解释 Triton 的 mask 如何保护尾部
   - ✓ 能说明 `blockDim.x` 与 `BLOCK_SIZE` 不是对应参数

---

## 附录：快速命令参考

### 完整实验流程

```bash
# 1. 环境准备
cd code/part2-kernels
uv sync
source ./activate-rocm.sh

# 2. 设置 commit
export SOURCE_COMMIT="$(git rev-parse HEAD)"

# 3. 运行 benchmark
bash chapter8/run_all.sh

# 4. 运行 profiling
bash chapter8/profile_all.sh

# 5. 查看结果
cat chapter8/evidence/summary.csv
cat chapter8/evidence/profile_summary.csv
```

### 单独运行某个版本

```bash
# HIP v0
./chapter8/build/vector_add_hip --version v0 --size 16777216 --block 256

# Triton t0
python chapter8/vector_add_triton.py --version t0 --size 16777216 --block 256
```

### 边界测试

```bash
# 测试 N=1027（不能被 4 整除）
./chapter8/build/vector_add_hip --version v3 --size 1027 --block 256 --warmup 0 --repeat 1
```

### Triton 可视化

```bash
# 生成 trace（不启动服务）
python chapter8/visualize_triton.py --size 13 --block 8

# 启动实时可视化
python chapter8/visualize_triton.py --size 13 --block 8 --launch
# 浏览器打开 http://127.0.0.1:5001
```

### 修改参数重跑

```bash
# 改变 shape
export SIZE=8388608
bash chapter8/run_all.sh

# 改变 block size
export HIP_BLOCK=512
export TRITON_BLOCK=512
bash chapter8/run_all.sh

# 改变 GPU 架构（需要重新编译）
export GPU_ARCH=gfx1100
bash chapter8/run_all.sh
```

---

## 下一步

- **第 9 章 Reduction**：撤掉"每个输出彼此独立"的前提，学习跨线程通信、LDS 与 Wave Shuffle
- **练习**：把 Vector Add 改成 ReLU、Scale & Bias，验证模板可迁移性
- **深入**：阅读 AMD HIP Performance Guidelines，理解合并访存的硬件机制

---

**文档版本**：2026-07-31  
**对应源码**：`hello-gpu/code/part2-kernels/chapter8/`  
**参考平台**：AMD Radeon RX 9070 XT (gfx1201) + ROCm 7.13 + Ubuntu 24.04